### Explanation of Each Column in the Used Bikes Dataset

1. **brand**
   - The manufacturer or company of the bike (e.g., Honda, Yamaha). Identifies the make of each bike, which is crucial for analysis and comparison. Brand reputation and popularity can significantly influence the price, demand, and resale value.

2. **bike_name**
   - The specific model name of the bike.
   - To distinguish between different models under the same brand.
   - Certain models may be more desirable, rare, or have better features, impacting their market value.

3. **year**
   - The manufacturing year of the bike.
   - To indicate the age of the bike, which is a key factor in valuation.
   - Older bikes generally depreciate more, while newer bikes retain higher value.

4. **price**
   - The listed selling price of the used bike.
   - It is the main variable of interest for buyers and sellers.
   - Price is influenced by all other columns and is used for market analysis and trend identification.

5. **kms_driven**
   - The total kilometers the bike has been driven.
   - Indicates the usage and wear of the bike.
   - Higher kilometers usually mean more wear and lower price; lower kilometers can command a premium.

6. **power**
   - The engine power of the bike, typically measured in bhp (brake horsepower).
   - To reflect the performance capability of the bike.
   - Bikes with higher power are often more expensive and appeal to performance-oriented buyers.

7. **owner_type**
   - Indicates whether the bike is first, second, or third hand, etc.
   - Ownership history affects buyer trust and resale value.
   - First-owner bikes are generally more valuable than those with multiple previous owners.

8. **location**
   - The city or region where the bike is being sold.
   - Location can affect demand, price, and availability.
   - Bikes in metropolitan areas may have higher prices due to higher demand; regional preferences may also exist.

In [0]:
from pyspark.sql import SparkSession 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [0]:
spark = SparkSession.builder.appName("Used_Bikes_Analysis").getOrCreate()

In [0]:
data = spark.read.csv('/Volumes/workspace/default/used_bikes_analysis/Used_Bikes.csv', header=True, inferSchema=True)

In [0]:
data.head(5)

In [0]:
display(data)

In [0]:
data.show()

In [0]:
data.head()

In [0]:
data.printSchema()

### First 5 rows of data

In [0]:
data.show(5)

In [0]:
data.describe()

### Statistics of the used_bikes

In [0]:
display(data.describe())

### Grouping Data by Brand, so we'll get the number of bikes for each brand

In [0]:
data.groupBy('brand').count().show()

In [0]:
from pyspark.sql.functions import count, col

# Get top 10 brands by bike count
brand_percent = data.groupBy('brand') \
    .agg(count('*').alias('bikes_count')) \
    .orderBy(col('bikes_count'), ascending=False) \
    .limit(10)

# Convert to Pandas for plotting
brand_percent_pd = brand_percent.toPandas()

import matplotlib.pyplot as plt

plt.pie(brand_percent_pd['bikes_count'], labels=brand_percent_pd['brand'], autopct='%1.1f%%')
plt.title('Bikes percent by top 10 brands')
plt.show()

### Let's check how many models in each brand and how many bikes in each model

In [0]:
from pyspark.sql.functions import count, col, countDistinct

# Count bikes per brand and model
bikes_per_model = data.groupBy('brand', 'bike_name').agg(count('*').alias('bikes_count'))

# Count distinct models per brand
models_per_brand = data.groupBy('brand').agg(countDistinct('bike_name').alias('distinct_models'))

# Join them together
result = bikes_per_model.join(models_per_brand, on='brand', how='left')

display(result.orderBy(col('bikes_count').desc()))

### Knowing the average price of each used model

In [0]:
from pyspark.sql.functions import avg, round, col

display(
    data.groupBy('brand', 'bike_name')
        .agg(round(avg('price'), 0).alias('avg_price'))
        .orderBy(col('avg_price'), ascending=False)
)

### More than 10 lakh priced by model

In [0]:
from pyspark.sql.functions import avg, round

morethan_1M_bikes = data.groupBy('brand', 'bike_name') \
        .agg(round(avg('price'), 0).alias('avg_price')) \
        .where(col('avg_price') > 1000000) \
        .orderBy('avg_price', ascending=False)
display(morethan_1M_bikes)

### Between 5-10 lakh priced model

In [0]:
between_500K_1M_bikes = data.groupBy('brand', 'bike_name') \
        .agg(round(avg('price'), 0).alias('avg_price')) \
        .where((col('avg_price') <= 1000000) & (col('avg_price') > 500000)) \
        .orderBy(col('avg_price'), ascending=False)
display(between_500K_1M_bikes)

### Between 1-5 lakh priced models

In [0]:
between_100K_500K_bikes = data.groupBy('brand', 'bike_name') \
        .agg(round(avg('price'), 0).alias('avg_price')) \
        .where((col('avg_price') <= 500000) & (col('avg_price') > 100000)) \
        .orderBy(col('avg_price'), ascending=False)

display(between_100K_500K_bikes)

### 50K - 1 lakh priced models

In [0]:
between_100K_50K_bikes = data.groupBy('brand', 'bike_name') \
        .agg(round(avg('price'), 0).alias('avg_price')) \
        .where((col('avg_price') <= 100000) & (col('avg_price') > 50000)) \
        .orderBy(col('avg_price'), ascending=False)
display(between_100K_50K_bikes)

### 50K and less priced models

In [0]:
below_50K_bikes = data.groupBy('brand', 'bike_name') \
        .agg(round(avg('price'), 0).alias('avg_price')) \
        .where((col('avg_price') <= 50000)) \
        .orderBy(col('avg_price'), ascending=False)
display(below_50K_bikes)

In [0]:
least_priced_top_10_below_50K_bikes = below_50K_bikes \
                    .orderBy(col('avg_price')) \
                    .limit(10)
least_priced_top_10_between_100K_50K_bikes = between_100K_50K_bikes \
                    .orderBy(col('avg_price')) \
                    .limit(10)
least_priced_top_10_between_100K_500K_bikes = between_100K_500K_bikes \
                    .orderBy(col('avg_price')) \
                    .limit(10)
least_priced_top_10_between_500K_1M_bikes = between_500K_1M_bikes \
                    .orderBy(col('avg_price')) \
                    .limit(10)
least_priced_top_10_morethan_1M_bikes = morethan_1M_bikes \
                    .orderBy(col('avg_price')) \
                    .limit(10)

In [0]:
most_priced_top_10_below_50K_bikes = below_50K_bikes \
                    .orderBy(col('avg_price'), ascending=False) \
                    .limit(10)
most_priced_top_10_between_100K_50K_bikes = between_100K_50K_bikes \
                    .orderBy(col('avg_price'), ascending=False) \
                    .limit(10)
most_priced_top_10_between_100K_500K_bikes = between_100K_500K_bikes \
                    .orderBy(col('avg_price'), ascending=False) \
                    .limit(10)
most_priced_top_10_between_500K_1M_bikes = between_500K_1M_bikes \
                    .orderBy(col('avg_price'), ascending=False) \
                    .limit(10)
most_priced_top_10_morethan_1M_bikes = morethan_1M_bikes \
                    .orderBy(col('avg_price'), ascending=False) \
                    .limit(10)

### Let's visualize all brands in one grid 

In [0]:
under_50K = below_50K_bikes.toPandas()
btw_50K_100K = between_100K_50K_bikes.toPandas()
btw_100K_500K = between_100K_500K_bikes.toPandas()
btw_500K_1M = between_500K_1M_bikes.toPandas()
morethan_1M = morethan_1M_bikes.toPandas()

In [0]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(20, 15))

brand_dfs = [under_50K, btw_50K_100K, btw_100K_500K, btw_500K_1M, morethan_1M]
titles = [
    'Bikes Under 50K by Brand',
    'Bikes between 50K and 100K by Brand',
    'Bikes between 100K and 500K by Brand',
    'Bikes 500K and 1M by Brand',
    'Bikes more than 1M by Brand'
]

for idx, (df, title) in enumerate(zip(brand_dfs, titles)):
    row, col = divmod(idx, 3)
    axes[row, col].bar(df['brand'], df['avg_price'])
    axes[row, col].set_xlabel('Brand')
    axes[row, col].set_ylabel('avg_price')
    axes[row, col].set_title(title)
    axes[row, col].tick_params(axis='x', rotation=45)

fig.delaxes(axes[1, 2])  # Remove unused subplot

plt.tight_layout()
plt.show()

### Sales analysis for bikes low under each category

In [0]:
least_priced_top_10_below_50K = least_priced_top_10_below_50K_bikes.toPandas()
least_priced_top_10_between_100K_50K = least_priced_top_10_between_100K_50K_bikes.toPandas()
least_priced_top_10_between_100K_500K = least_priced_top_10_between_100K_500K_bikes.toPandas()
least_priced_top_10_between_500K_1M = least_priced_top_10_between_500K_1M_bikes.toPandas()
least_priced_top_10_morethan_1M = least_priced_top_10_morethan_1M_bikes.toPandas()

In [0]:

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(20, 15))

least_priced_top_10_bikes = [
                least_priced_top_10_below_50K,
                least_priced_top_10_between_100K_50K,
                least_priced_top_10_between_100K_500K,
                least_priced_top_10_between_500K_1M,
                least_priced_top_10_morethan_1M
]

titles = [
    'Bikes Under 50K by Bike',
    'Bikes between 50K and 100K by Bike',
    'Bikes between 100K and 500K by Bike',
    'Bikes 500K and 1M by Bike',
    'Bikes more than 1M by Bike'
]
color = [
    '#00ff00', '#ffff00', '#ffbf00', '#ff8000', '#ff0000'
]

for idx, (df, title) in enumerate(zip(least_priced_top_10_bikes, titles)):
    row, col = divmod(idx, 3)
    axes[row, col].bar(df['bike_name'], df['avg_price'], color=color[idx])
    axes[row, col].set_xlabel('Bike')
    axes[row, col].set_ylabel('avg_price')
    axes[row, col].set_title(title)
    axes[row, col].tick_params(axis='x', rotation=90)

fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.title('Least Priced Top 10 Bikes by Price')
plt.show()


### Sales Analysis for Bikes Priced High Under Each Category

In [0]:
most_priced_top_10_below_50K = most_priced_top_10_below_50K_bikes.toPandas()
most_priced_top_10_between_100K_50K = most_priced_top_10_between_100K_50K_bikes.toPandas()
most_priced_top_10_between_100K_500K = most_priced_top_10_between_100K_500K_bikes.toPandas()
most_priced_top_10_between_500K_1M = most_priced_top_10_between_500K_1M_bikes.toPandas()
most_priced_top_10_morethan_1M = most_priced_top_10_morethan_1M_bikes.toPandas()

In [0]:
fig, axes = plt.subplots(2, 3, figsize=(20, 15))

most_priced_top_10_bikes = [
                            most_priced_top_10_below_50K,
                            most_priced_top_10_between_100K_50K,
                            most_priced_top_10_between_100K_500K,
                            most_priced_top_10_between_500K_1M,
                            most_priced_top_10_morethan_1M
]

titles = [
    'Bikes Under 50K by Bike',
    'Bikes between 50K and 100K by Bike',
    'Bikes between 100K and 500K by Bike',
    'Bikes 500K and 1M by Bike',
    'Bikes more than 1M by Bike'
]
color = [
    '#00ff00', '#ffff00', '#ffbf00', '#ff8000', '#ff0000'
]

for idx, (df, title) in enumerate(zip(most_priced_top_10_bikes, titles)):
    row, col = divmod(idx, 3)
    axes[row, col].bar(df['bike_name'], df['avg_price'], color=color[idx])
    axes[row, col].set_xlabel('Bike')
    axes[row, col].set_ylabel('avg_price')
    axes[row, col].set_title(title)
    axes[row, col].tick_params(axis='x', rotation=90)

fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.title('Most Priced Top 10 Bikes by Price')
plt.show()

- I need the top 10 least priced bikes.
- I need to know what are the best bikes being sold in that category.
- I need to find the best mileage bikes in that price category.
- I want to know if lower-priced bikes are being sold mostly.
- I need to predict the price if the user provides the bike's year of manufacturing, brand, and the number of kilometers it has been ridden (i.e., if the user is selling that bike, predict its actual resale price).

In [0]:
data.show(5)

In [0]:

current_year = 2026
data_pd = data.select('age', 'price', 'kms_driven').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scatter plot: Age vs Price
axes[0].scatter(data_pd['age'], data_pd['price'], alpha=0.6)
axes[0].set_xlabel('Age of Bike (years)')
axes[0].set_ylabel('Price')
axes[0].set_title('Bike Age vs Price')

# Scatter plot: Kilometers Driven vs Price
axes[1].scatter(data_pd['kms_driven'], data_pd['price'], alpha=0.8)
axes[1].set_xlabel('Kilometers Driven')
axes[1].set_ylabel('Price')
axes[1].set_title('Kilometers Driven vs Price')

plt.tight_layout()
plt.show()

- There is a clear negative correlation between the age of the bike and its price.
- As the age of the bike increases, the price decreases.
- Similarly, as the number of kilometers driven increases, the price also decreases.


In [0]:
data_pd = data.select("power", "price").toPandas()
plt.scatter(data_pd['power'], data_pd['price'], alpha=0.8, marker='*', color='red')
plt.xlabel('Power')
plt.ylabel('Price')
plt.title('Power vs Price')
plt.show()

### Let's prepare the data for predictions

In [0]:
data.show(5)

In [0]:
from pyspark.sql.functions import sum, col

null_exists = data.select([sum(col(c).isNull().cast("int")).alias(c) for c in data.columns])
null_exists.show()

In [0]:
display(data.select('owner').distinct())

###### Here we've columns:
    *Price:* Dependent feature which means price depends on brand, power, age, kms_driven
    *Other colums:* Independent features (bike_name, city, kms_driven, owner, age, power, brand)

#### String columns Enconding:
######** In data table, there are columns whose type is string. 
######** Machine learning models works on only numbers. 
######** So somehow we need to convert that string types into numerical. 
######** This technique is being called as Encoding.
######** There are numerous encoding techniques based on our usage.

In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

# String columns to encode
string_cols = ['brand', 'bike_name', 'city', 'owner']

# Create StringIndexers for each categorical column
indexers = [StringIndexer(inputCol=col, outputCol=f"encoded{col}") for col in string_cols]

# Create pipeline and transform data
pipeline = Pipeline(stages=indexers)
data_encoded = pipeline.fit(data).transform(data)

data_encoded.show(5)

In [0]:
from pyspark.ml.feature import VectorAssembler

# Select independent features (exclude price)
independent_cols = ['encodedbrand', 'encodedbike_name', 'encodedcity', 'kms_driven', 'encodedowner', 'age', 'power']

# Create VectorAssembler to combine all independent features into a single vector
vecAssembler = VectorAssembler(inputCols=independent_cols, outputCol="features")

In [0]:
# Transform data to add features vector column
data_vectorized = vecAssembler.transform(data_encoded)

# Select only features (independent) and price (dependent) columns for ML
ml_data = data_vectorized.select('features', 'price')

ml_data.show(5, truncate=False)

In [0]:
# Display schema to verify features vector and price column
ml_data.printSchema()
print(f"Total records: {ml_data.count()}")

In [0]:
from pyspark.ml.regression import LinearRegression

train_data, test_data = ml_data.randomSplit([0.7, 0.3], seed=42)

lr = LinearRegression(featuresCol='features', labelCol='price', maxIter=10, regParam=0.3)
lr_model = lr.fit(train_data)
# Make predictions on test data
predictions = lr_model.transform(test_data)

# Display predictions and actual values
display(predictions.select('prediction', 'price', 'features'))
from pyspark.ml.evaluation import RegressionEvaluator

# Create RegressionEvaluator to evaluate model performance
evaluator = RegressionEvaluator(labelCol='price', predictionCol='prediction', metricName='rmse')

# Evaluate model on test data
rmse = evaluator.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE): {rmse}")

In [0]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Train-test split (already done above)
# train_data, test_data = ml_data.randomSplit([0.7, 0.3], seed=42)

# Initialize Random Forest Regressor
rf = RandomForestRegressor(featuresCol='features', labelCol='price', numTrees=100, maxDepth=5, maxBins=500, seed=42)
rf_model = rf.fit(train_data)

# Make predictions
rf_predictions = rf_model.transform(test_data)

# Display predictions and actual values
display(rf_predictions.select('prediction', 'price', 'features'))

# Evaluate model performance with different metrics
evaluator_rmse = RegressionEvaluator(labelCol='price', predictionCol='prediction', metricName='rmse')
evaluator_mae = RegressionEvaluator(labelCol='price', predictionCol='prediction', metricName='mae')
evaluator_r2 = RegressionEvaluator(labelCol='price', predictionCol='prediction', metricName='r2')

rmse_rf = evaluator_rmse.evaluate(rf_predictions)
mae_rf = evaluator_mae.evaluate(rf_predictions)
r2_rf = evaluator_r2.evaluate(rf_predictions)

print(f"Random Forest RMSE: {rmse_rf}")
print(f"Random Forest MAE: {mae_rf}")
print(f"Random Forest R2: {r2_rf}")

# Confusion matrix is not applicable for regression, but you can plot prediction vs actual
import pandas as pd
import matplotlib.pyplot as plt

rf_pred_pd = rf_predictions.select('prediction', 'price').toPandas()
plt.figure(figsize=(6,6))
plt.scatter(rf_pred_pd['price'], rf_pred_pd['prediction'], alpha=0.5)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Random Forest Regression: Actual vs Predicted')
plt.plot([rf_pred_pd['price'].min(), rf_pred_pd['price'].max()],
         [rf_pred_pd['price'].min(), rf_pred_pd['price'].max()], 'r--')
plt.show()